In [2]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ==========================================
# 1. FROM-SCRATCH OPTIMIZER IMPLEMENTATIONS
# ==========================================

class Optimizer:
    def __init__(self, lr=0.01):
        self.lr = lr

class SGD(Optimizer):
    def update(self, params, grads):
        for key in params.keys():
            params[key] -= self.lr * grads[key]

class Momentum(Optimizer):
    def __init__(self, lr=0.01, momentum=0.9):
        super().__init__(lr)
        self.momentum = momentum
        self.v = None

    def update(self, params, grads):
        if self.v is None:
            self.v = {key: np.zeros_like(val) for key, val in params.items()}
        for key in params.keys():
            self.v[key] = self.momentum * self.v[key] + self.lr * grads[key]
            params[key] -= self.v[key]

class Adagrad(Optimizer):
    def __init__(self, lr=0.01, epsilon=1e-8):
        super().__init__(lr)
        self.epsilon = epsilon
        self.G = None

    def update(self, params, grads):
        if self.G is None:
            self.G = {key: np.zeros_like(val) for key, val in params.items()}
        for key in params.keys():
            self.G[key] += grads[key] ** 2
            params[key] -= (self.lr / np.sqrt(self.G[key] + self.epsilon)) * grads[key]

class RMSprop(Optimizer):
    def __init__(self, lr=0.001, beta=0.9, epsilon=1e-8):
        super().__init__(lr)
        self.beta = beta
        self.epsilon = epsilon
        self.E = None

    def update(self, params, grads):
        if self.E is None:
            self.E = {key: np.zeros_like(val) for key, val in params.items()}
        for key in params.keys():
            self.E[key] = self.beta * self.E[key] + (1 - self.beta) * (grads[key] ** 2)
            params[key] -= (self.lr / np.sqrt(self.E[key] + self.epsilon)) * grads[key]

class Adadelta(Optimizer):
    def __init__(self, rho=0.95, epsilon=1e-6):
        super().__init__(1.0) # lr is generally ignored/set to 1
        self.rho = rho
        self.epsilon = epsilon
        self.E_g = None
        self.E_dx = None

    def update(self, params, grads):
        if self.E_g is None:
            self.E_g = {key: np.zeros_like(val) for key, val in params.items()}
            self.E_dx = {key: np.zeros_like(val) for key, val in params.items()}

        for key in params.keys():
            self.E_g[key] = self.rho * self.E_g[key] + (1 - self.rho) * (grads[key] ** 2)
            dx = np.sqrt((self.E_dx[key] + self.epsilon) / (self.E_g[key] + self.epsilon)) * grads[key]
            self.E_dx[key] = self.rho * self.E_dx[key] + (1 - self.rho) * (dx ** 2)
            params[key] -= dx

class Adam(Optimizer):
    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):
        super().__init__(lr)
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.m = None
        self.v = None
        self.t = 0

    def update(self, params, grads):
        if self.m is None:
            self.m = {key: np.zeros_like(val) for key, val in params.items()}
            self.v = {key: np.zeros_like(val) for key, val in params.items()}

        self.t += 1
        for key in params.keys():
            self.m[key] = self.beta1 * self.m[key] + (1 - self.beta1) * grads[key]
            self.v[key] = self.beta2 * self.v[key] + (1 - self.beta2) * (grads[key] ** 2)

            m_hat = self.m[key] / (1 - self.beta1 ** self.t)
            v_hat = self.v[key] / (1 - self.beta2 ** self.t)

            params[key] -= self.lr * m_hat / (np.sqrt(v_hat) + self.epsilon)

class NAdam(Optimizer):
    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):
        super().__init__(lr)
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.m = None
        self.v = None
        self.t = 0

    def update(self, params, grads):
        if self.m is None:
            self.m = {key: np.zeros_like(val) for key, val in params.items()}
            self.v = {key: np.zeros_like(val) for key, val in params.items()}

        self.t += 1
        for key in params.keys():
            self.m[key] = self.beta1 * self.m[key] + (1 - self.beta1) * grads[key]
            self.v[key] = self.beta2 * self.v[key] + (1 - self.beta2) * (grads[key] ** 2)

            m_hat = self.m[key] / (1 - self.beta1 ** self.t)
            v_hat = self.v[key] / (1 - self.beta2 ** self.t)

            # Incorporating Nesterov momentum directly into the Adam update
            m_hat_nesterov = self.beta1 * m_hat + (1 - self.beta1) * grads[key] / (1 - self.beta1 ** self.t)

            params[key] -= self.lr * m_hat_nesterov / (np.sqrt(v_hat) + self.epsilon)

# ==========================================
# 2. FROM-SCRATCH NEURAL NETWORK (MLP)
# ==========================================

class NeuralNetwork:
    def __init__(self, input_size, hidden_size, output_size):
        # He Initialization for deeper networks
        self.params = {
            'W1': np.random.randn(input_size, hidden_size) * np.sqrt(2. / input_size),
            'b1': np.zeros((1, hidden_size)),
            'W2': np.random.randn(hidden_size, output_size) * np.sqrt(2. / hidden_size),
            'b2': np.zeros((1, output_size))
        }
        self.cache = {}

    def relu(self, Z):
        return np.maximum(0, Z)

    def relu_derivative(self, Z):
        return Z > 0

    def softmax(self, Z):
        expZ = np.exp(Z - np.max(Z, axis=1, keepdims=True)) # Stabilized softmax
        return expZ / np.sum(expZ, axis=1, keepdims=True)

    def forward(self, X):
        self.cache['X'] = X
        self.cache['Z1'] = np.dot(X, self.params['W1']) + self.params['b1']
        self.cache['A1'] = self.relu(self.cache['Z1'])
        self.cache['Z2'] = np.dot(self.cache['A1'], self.params['W2']) + self.params['b2']
        self.cache['A2'] = self.softmax(self.cache['Z2'])
        return self.cache['A2']

    def compute_loss(self, Y, Y_pred):
        m = Y.shape[0]
        # Cross-Entropy Loss
        log_probs = -np.log(Y_pred[range(m), np.argmax(Y, axis=1)] + 1e-8)
        loss = np.sum(log_probs) / m
        return loss

    def backward(self, Y):
        m = Y.shape[0]
        grads = {}

        # Output layer gradients (Softmax + Cross Entropy simplification)
        dZ2 = self.cache['A2'] - Y
        grads['W2'] = np.dot(self.cache['A1'].T, dZ2) / m
        grads['b2'] = np.sum(dZ2, axis=0, keepdims=True) / m

        # Hidden layer gradients
        dA1 = np.dot(dZ2, self.params['W2'].T)
        dZ1 = dA1 * self.relu_derivative(self.cache['Z1'])
        grads['W1'] = np.dot(self.cache['X'].T, dZ1) / m
        grads['b1'] = np.sum(dZ1, axis=0, keepdims=True) / m

        return grads

# ==========================================
# 3. DATA ACQUISITION & TRAINING SCRIPT
# ==========================================

def get_data():
    print("Fetching Digits dataset via scikit-learn API...")
    digits = load_digits()
    X, y = digits.data, digits.target

    # Scale Data
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # One-Hot Encode Targets
    encoder = OneHotEncoder(sparse_output=False)
    y_encoded = encoder.fit_transform(y.reshape(-1, 1))

    return train_test_split(X, y_encoded, y, test_size=0.2, random_state=42)

def train_and_evaluate():
    # === FIXED LINE HERE ===
    # train_test_split returns 6 values when passed 3 arrays (X, y_encoded, y)
    X_train, X_test, y_train_enc, y_test_enc, y_train_raw, y_test = get_data()
    # =======================

    input_size = X_train.shape[1]
    hidden_size = 64
    output_size = 10
    epochs = 2
    batch_size = 32

    # Initialize all optimizers to compare
    optimizers_to_test = {
        'SGD': SGD(lr=0.01),
        'Momentum': Momentum(lr=0.01, momentum=0.9),
        'Adagrad': Adagrad(lr=0.01),
        'RMSprop': RMSprop(lr=0.001),
        'Adadelta': Adadelta(),
        'Adam': Adam(lr=0.001),
        'NAdam': NAdam(lr=0.001)
    }

    for name, optimizer in optimizers_to_test.items():
        print(f"\n{'='*50}")
        print(f"Training with Optimizer: {name}")
        print(f"{'='*50}")

        # Create a fresh network for each optimizer
        model = NeuralNetwork(input_size, hidden_size, output_size)

        # Training Loop
        for epoch in range(epochs):
            permutation = np.random.permutation(X_train.shape[0])
            X_train_shuffled = X_train[permutation]
            y_train_shuffled = y_train_enc[permutation]

            epoch_loss = 0
            num_batches = X_train.shape[0] // batch_size

            for i in range(0, X_train.shape[0], batch_size):
                X_batch = X_train_shuffled[i:i+batch_size]
                Y_batch = y_train_shuffled[i:i+batch_size]

                # Forward Pass
                Y_pred = model.forward(X_batch)

                # Calculate Loss
                loss = model.compute_loss(Y_batch, Y_pred)
                epoch_loss += loss

                # Backward Pass
                grads = model.backward(Y_batch)

                # Update Weights
                optimizer.update(model.params, grads)

            print(f"Epoch {epoch+1}/{epochs} - Average Loss: {epoch_loss / num_batches:.4f}")

        # Evaluation after 2 epochs
        print(f"\n--- Metrics for {name} ---")
        y_pred_probs = model.forward(X_test)
        y_pred_classes = np.argmax(y_pred_probs, axis=1)

        acc = accuracy_score(y_test, y_pred_classes)
        print(f"Final Accuracy: {acc * 100:.2f}%")

        print("\nClassification Report:")
        print(classification_report(y_test, y_pred_classes, zero_division=0))

        print("Confusion Matrix:")
        print(confusion_matrix(y_test, y_pred_classes))

if __name__ == "__main__":
    train_and_evaluate()

Fetching Digits dataset via scikit-learn API...

Training with Optimizer: SGD
Epoch 1/2 - Average Loss: 2.7691
Epoch 2/2 - Average Loss: 1.9201

--- Metrics for SGD ---
Final Accuracy: 45.83%

Classification Report:
              precision    recall  f1-score   support

           0       0.74      0.85      0.79        33
           1       0.19      0.25      0.22        28
           2       0.30      0.48      0.37        33
           3       0.20      0.03      0.05        34
           4       0.42      0.41      0.42        46
           5       0.33      0.15      0.21        47
           6       0.84      0.89      0.86        35
           7       0.45      0.76      0.57        34
           8       0.29      0.17      0.21        30
           9       0.51      0.62      0.56        40

    accuracy                           0.46       360
   macro avg       0.43      0.46      0.43       360
weighted avg       0.43      0.46      0.43       360

Confusion Matrix:
[[28  0